### Experiment with gpt 4.o-mini

In [1]:
import pandas as pd
import libs.prompts as prompts
from libs.exp_utils import classify_dataset, evaluate_experiment
from libs.openai import call_openai
from tqdm import tqdm
import json
import os

In [2]:
# Define model name
MODEL_NAME = "gpt-4o-mini"

### Dataset load

In [3]:
# Load ground truth bug locations
with open("../../Datasets/bugLocationDappScan.json", "r") as f:
    bug_locations = json.load(f)

In [4]:
def normalize_path(path):
    return path.replace('\\', '/').replace('./', '').replace('.\\', '') 

bugloc_dict = {normalize_path(item['file']): normalize_path(item['location']) for item in bug_locations}

csv_path = "../../Datasets/dw.csv"
df = pd.read_csv(csv_path)
df['has_error'] = df['has_error'].astype(bool)
subset = df.reset_index(drop=True)
subset.head()

,dataset_name,file_path,category,file_name,has_error
0,openZeppelin,openzeppelin-contracts-master/contracts/mocks/...,contracts,RegressionImplementation.sol,False
1,openZeppelin,openzeppelin-contracts-master/contracts/mocks/...,contracts,CallReceiverMock.sol,False
2,openZeppelin,openzeppelin-contracts-master/contracts/mocks/...,contracts,InitializableMock.sol,False
3,openZeppelin,openzeppelin-contracts-master/contracts/mocks/...,contracts,AccessManagedTarget.sol,False
4,openZeppelin,openzeppelin-contracts-master/contracts/mocks/...,contracts,TransientSlotMock.sol,False


#### Zero-shot Prompting

In [ ]:
results_df = classify_dataset(subset, prompts.create_zeroshot_locate_prompt, MODEL_NAME)
precision, recall, f1, conf_matrix = evaluate_experiment(results_df)
print(f"Precision: {precision:.2f}")
print(f"Recall: {recall:.2f}")
print(f"F1-Score: {f1:.2f}")
print(f"Confusion Matrix:\n{conf_matrix}")
results_df.head()

400it [12:23,  1.86s/it]

Precision: 56.56
Recall: 90.50
F1-Score: 69.62
Confusion Matrix:
[[ 61 139]
 [ 19 181]]


,predicted_has_error,actual_has_error,bug_type,location
0,1,0,Reentrancy vulnerability,Line 43
1,1,0,Reentrancy,NONE
2,1,0,Constructor Initialization Order Vulnerability,"Line 92, 100"
3,1,0,Storage Manipulation,20
4,0,0,NONE,NONE


In [ ]:
results_df['file_path'] = subset['file_path']
results_df['file_name'] = subset['file_name'] 

error_cases = results_df[(results_df['predicted_has_error'] == True) & (results_df['actual_has_error'] == True)].copy()

error_cases.head()



,predicted_has_error,actual_has_error,bug_type,location,file_path,file_name
200,1,1,SWC-114-Transaction Order Dependence,53,DAppSCAN-main/DAppSCAN-source/contracts/QuillA...,ERC20Permit.sol
202,1,1,SWC-103-Floating Pragma,4,DAppSCAN-main/DAppSCAN-source/contracts/Chains...,AccessControl.sol
203,1,1,SWC-101-Integer Overflow and Underflow,L90,DAppSCAN-main/DAppSCAN-source/contracts/openze...,HackerGold.sol
204,1,1,SWC-101-Integer Overflow and Underflow,122,DAppSCAN-main/DAppSCAN-source/contracts/PeckSh...,UserConfiguration.sol
206,1,1,SWC-107-Reentrancy,182-240,DAppSCAN-main/DAppSCAN-source/contracts/PeckSh...,FlashLoanLogic.sol


In [ ]:
def get_gt_category(row):
    file_path = row['file_path'].replace('DAppSCAN-main/DAppSCAN-source/', '')
    if file_path in bugloc_dict:
        location_json_path = os.path.join('../../Datasets/DAppSCAN-main/DAppSCAN-source/', bugloc_dict[file_path]) #.replace('.sol', '.json'))
        if os.path.exists(location_json_path):
            with open(location_json_path, 'r') as f:
                location_data = json.load(f)
                swcs = location_data.get('SWCs', [])
                categories = [swc['category'] for swc in swcs if 'category' in swc]
                return categories if categories else ['Unknown']
        else:
            return ['JSON não encontrado']
    else:
        return ['Arquivo não encontrado']


In [ ]:
llm_categories = []
gt_categories = []

for idx, row in error_cases.iterrows():
    print(f"\n--- Rodando para idx {idx} ---")
    gt_cat = get_gt_category(row)

    llm_categories.append(row['bug_type'])
    llm_categories.append(row['location'])
    gt_categories.append(gt_cat)
    print(f"File: {row['file_path']}\nLLM: {row['bug_type']}\nGT: {gt_cat}\n")

    print("LOCATION: LINE - ", row['location'])



--- Rodando para idx 200 ---
File: DAppSCAN-main/DAppSCAN-source/contracts/QuillAudits-1inch-token/1inch-token-99fd056f91005ca521a02a005f7bcd8f77e06afc/contracts/ERC20Permit.sol
LLM: SWC-114-Transaction Order Dependence
GT: ['SWC-114-Transaction Order Dependence']

LOCATION: LINE -  53

--- Rodando para idx 202 ---
File: DAppSCAN-main/DAppSCAN-source/contracts/Chainsulting-GSPI Club-project2/openzeppelin-contracts-3.2.0/contracts/access/AccessControl.sol
LLM: SWC-103-Floating Pragma
GT: ['SWC-103-Floating Pragma', 'SWC-102-Outdated Compiler Version']

LOCATION: LINE -  4

--- Rodando para idx 203 ---
File: DAppSCAN-main/DAppSCAN-source/contracts/openzeppelin-EtherCamp_Hacker_Gold/virtual-accelerator-2529ffe5efd5294b44f1bc89dc9a4721a7b16409/HackerGold.sol
LLM: SWC-101-Integer Overflow and Underflow
GT: ['SWC-101-Integer Overflow and Underflow', 'SWC-135-Code With No Effects', 'SWC-135-Code With No Effects']

LOCATION: LINE -  L90

--- Rodando para idx 204 ---
File: DAppSCAN-main/DAppSC

In [ ]:
comparison_results = []

for bug_pred, gt_cat in tqdm(zip(error_cases['bug_type'], gt_categories), total=len(gt_categories)):
    prompt = prompts.create_zeroshot_error_comparison_prompt(bug_pred, gt_cat)
    response = call_openai(MODEL_NAME, prompt)
    try:
        result = json.loads(response)
        same_error = int(result.get("same_error", False))
    except Exception as e:
        print("Erro ao interpretar resposta da LLM:", e)
        same_error = 0 

    comparison_results.append({'actual_has_error': 1, 'predicted_has_error': same_error})

comparison_df = pd.DataFrame(comparison_results)

precision, recall, f1, conf_matrix = evaluate_experiment(comparison_df)
print(f"Precision: {precision:.2f}")
print(f"Recall: {recall:.2f}")
print(f"F1-Score: {f1:.2f}")
print(f"Confusion Matrix:\n{conf_matrix}")


100%|██████████| 181/181 [03:51<00:00,  1.28s/it]

Precision: 100.00
Recall: 81.77
F1-Score: 89.97
Confusion Matrix:
[[  0   0]
 [ 33 148]]


#### Zero-shot CoT Prompting

In [5]:
results_df = classify_dataset(subset, prompts.create_zeroshot_cot_locate_prompt, MODEL_NAME)
precision, recall, f1, conf_matrix = evaluate_experiment(results_df)
print(f"Precision: {precision:.2f}")
print(f"Recall: {recall:.2f}")
print(f"F1-Score: {f1:.2f}")
print(f"Confusion Matrix:\n{conf_matrix}")
results_df.head()

400it [36:11,  5.43s/it]

Precision: 54.42
Recall: 98.50
F1-Score: 70.11
Confusion Matrix:
[[ 35 165]
 [  3 197]]


,predicted_has_error,actual_has_error,bug_type
0,1,0,Fallback Function Vulnerability
1,1,0,Reentrancy Vulnerability
2,1,0,Constructor Initialization Order Vulnerability
3,1,0,Storage Manipulation Vulnerability
4,1,0,Reentrancy Vulnerability


In [6]:
results_df['file_path'] = subset['file_path']
results_df['file_name'] = subset['file_name'] 

error_cases = results_df[(results_df['predicted_has_error'] == True) & (results_df['actual_has_error'] == True)].copy()

error_cases.head()

,predicted_has_error,actual_has_error,bug_type,file_path,file_name
200,1,1,SWC-114-Transaction Order Dependence,DAppSCAN-main/DAppSCAN-source/contracts/QuillA...,ERC20Permit.sol
201,1,1,SWC-103-Floating Pragma,DAppSCAN-main/DAppSCAN-source/contracts/Chains...,AccessControl.sol
202,1,1,SWC-103-Floating Pragma,DAppSCAN-main/DAppSCAN-source/contracts/Chains...,AccessControl.sol
203,1,1,SWC-135-Code With No Effects,DAppSCAN-main/DAppSCAN-source/contracts/openze...,HackerGold.sol
204,1,1,SWC-101-Integer Overflow and Underflow,DAppSCAN-main/DAppSCAN-source/contracts/PeckSh...,UserConfiguration.sol


In [7]:
def get_gt_category(row):
    file_path = row['file_path'].replace('DAppSCAN-main/DAppSCAN-source/', '')
    if file_path in bugloc_dict:
        location_json_path = os.path.join('../../Datasets/DAppSCAN-main/DAppSCAN-source/', bugloc_dict[file_path])
        if os.path.exists(location_json_path):
            with open(location_json_path, 'r') as f:
                location_data = json.load(f)
                swcs = location_data.get('SWCs', [])
                categories = [swc['category'] for swc in swcs if 'category' in swc]
                return categories if categories else ['Unknown']
        else:
            return ['JSON não encontrado']
    else:
        return ['Arquivo não encontrado']

In [9]:
# Roda a checagem
llm_categories = []
gt_categories = []

for idx, row in error_cases.iterrows():
    print(f"\n--- Rodando para idx {idx} ---")
    gt_cat = get_gt_category(row)

    llm_categories.append(row['bug_type'])
    #llm_categories.append(row['location'])  # Assuming 'bug_type' is the LLM's prediction
    gt_categories.append(gt_cat)
    print(f"File: {row['file_path']}\nLLM: {row['bug_type']}\nGT: {gt_cat}\n")

    #print("LOCATION: LINE - ", row['location'])
#print("LLM Categories:", llm_categories, "GT Categories:", gt_categories)



--- Rodando para idx 200 ---
File: DAppSCAN-main/DAppSCAN-source/contracts/QuillAudits-1inch-token/1inch-token-99fd056f91005ca521a02a005f7bcd8f77e06afc/contracts/ERC20Permit.sol
LLM: SWC-114-Transaction Order Dependence
GT: ['SWC-114-Transaction Order Dependence']


--- Rodando para idx 201 ---
File: DAppSCAN-main/DAppSCAN-source/contracts/Chainsulting-SWAPP Protocol-project1/openzeppelin-contracts-3.3.0/contracts/access/AccessControl.sol
LLM: SWC-103-Floating Pragma
GT: ['SWC-103-Floating Pragma']


--- Rodando para idx 202 ---
File: DAppSCAN-main/DAppSCAN-source/contracts/Chainsulting-GSPI Club-project2/openzeppelin-contracts-3.2.0/contracts/access/AccessControl.sol
LLM: SWC-103-Floating Pragma
GT: ['SWC-103-Floating Pragma', 'SWC-102-Outdated Compiler Version']


--- Rodando para idx 203 ---
File: DAppSCAN-main/DAppSCAN-source/contracts/openzeppelin-EtherCamp_Hacker_Gold/virtual-accelerator-2529ffe5efd5294b44f1bc89dc9a4721a7b16409/HackerGold.sol
LLM: SWC-135-Code With No Effects
GT

In [10]:
comparison_results = []

for bug_pred, gt_cat in tqdm(zip(error_cases['bug_type'], gt_categories), total=len(gt_categories)):
    prompt = prompts.create_zeroshot_error_comparison_prompt(bug_pred, gt_cat)
    response = call_openai(MODEL_NAME, prompt)
    try:
        result = json.loads(response)
        same_error = int(result.get("same_error", False))
    except Exception as e:
        print("Erro ao interpretar resposta da LLM:", e)
        same_error = 0 

    comparison_results.append({'actual_has_error': 1, 'predicted_has_error': same_error})

comparison_df = pd.DataFrame(comparison_results)

precision, recall, f1, conf_matrix = evaluate_experiment(comparison_df)
print(f"Precision: {precision:.2f}")
print(f"Recall: {recall:.2f}")
print(f"F1-Score: {f1:.2f}")
print(f"Confusion Matrix:\n{conf_matrix}")

100%|██████████| 197/197 [04:09<00:00,  1.26s/it]

Precision: 100.00
Recall: 80.20
F1-Score: 89.01
Confusion Matrix:
[[  0   0]
 [ 39 158]]


#### Zero-shot ToT Prompting

In [11]:
results_df = classify_dataset(subset, prompts.create_zeroshot_tot_locate_prompt, MODEL_NAME)
precision, recall, f1, conf_matrix = evaluate_experiment(results_df)
print(f"Precision: {precision:.2f}")
print(f"Recall: {recall:.2f}")
print(f"F1-Score: {f1:.2f}")
print(f"Confusion Matrix:\n{conf_matrix}")
results_df.head()

248it [36:55, 25.78s/it]

Attempt 1 failed: Request timed out.
extract_json_from_text: recebeu texto vazio ou None!
Texto com erro: None


249it [37:56, 36.50s/it]

Attempt 1 failed: Request timed out.
extract_json_from_text: recebeu texto vazio ou None!
Texto com erro: None


400it [1:06:11,  9.93s/it]

Precision: 53.95
Recall: 100.00
F1-Score: 70.09
Confusion Matrix:
[[ 31 169]
 [  0 198]]


,predicted_has_error,actual_has_error,bug_type
0,1,0,Shadowing and State Modification in Fallback
1,1,0,"Revert without reason, Infinite loop"
2,1,0,Initializer misuse and inheritance conflict
3,1,0,Storage manipulation vulnerability
4,1,0,"Reentrancy, Lack of Access Control, Informatio..."


In [12]:
results_df['file_path'] = subset['file_path']
results_df['file_name'] = subset['file_name'] 

error_cases = results_df[(results_df['predicted_has_error'] == True) & (results_df['actual_has_error'] == True)].copy()

error_cases.head()

,predicted_has_error,actual_has_error,bug_type,file_path,file_name
200,1,1,SWC-114-Transaction Order Dependence,DAppSCAN-main/DAppSCAN-source/contracts/QuillA...,ERC20Permit.sol
201,1,1,"Floating Pragma, Access Control Vulnerability",DAppSCAN-main/DAppSCAN-source/contracts/Chains...,AccessControl.sol
202,1,1,Floating Pragma and Outdated Compiler Version,DAppSCAN-main/DAppSCAN-source/contracts/Chains...,AccessControl.sol
203,1,1,Integer Overflow and Underflow,DAppSCAN-main/DAppSCAN-source/contracts/openze...,HackerGold.sol
204,1,1,SWC-101-Integer Overflow and Underflow,DAppSCAN-main/DAppSCAN-source/contracts/PeckSh...,UserConfiguration.sol


In [13]:
def get_gt_category(row):
    file_path = row['file_path'].replace('DAppSCAN-main/DAppSCAN-source/', '')
    if file_path in bugloc_dict:
        location_json_path = os.path.join('../../Datasets/DAppSCAN-main/DAppSCAN-source/', bugloc_dict[file_path])
        if os.path.exists(location_json_path):
            with open(location_json_path, 'r') as f:
                location_data = json.load(f)
                swcs = location_data.get('SWCs', [])
                categories = [swc['category'] for swc in swcs if 'category' in swc]
                return categories if categories else ['Unknown']
        else:
            return ['JSON não encontrado']
    else:
        return ['Arquivo não encontrado']

In [14]:
# Roda a checagem
llm_categories = []
gt_categories = []

for idx, row in error_cases.iterrows():
    print(f"\n--- Rodando para idx {idx} ---")
    gt_cat = get_gt_category(row)

    llm_categories.append(row['bug_type'])
    #llm_categories.append(row['location'])  # Assuming 'bug_type' is the LLM's prediction
    gt_categories.append(gt_cat)
    print(f"File: {row['file_path']}\nLLM: {row['bug_type']}\nGT: {gt_cat}\n")

    #print("LOCATION: LINE - ", row['location'])
#print("LLM Categories:", llm_categories, "GT Categories:", gt_categories)



--- Rodando para idx 200 ---
File: DAppSCAN-main/DAppSCAN-source/contracts/QuillAudits-1inch-token/1inch-token-99fd056f91005ca521a02a005f7bcd8f77e06afc/contracts/ERC20Permit.sol
LLM: SWC-114-Transaction Order Dependence
GT: ['SWC-114-Transaction Order Dependence']


--- Rodando para idx 201 ---
File: DAppSCAN-main/DAppSCAN-source/contracts/Chainsulting-SWAPP Protocol-project1/openzeppelin-contracts-3.3.0/contracts/access/AccessControl.sol
LLM: Floating Pragma, Access Control Vulnerability
GT: ['SWC-103-Floating Pragma']


--- Rodando para idx 202 ---
File: DAppSCAN-main/DAppSCAN-source/contracts/Chainsulting-GSPI Club-project2/openzeppelin-contracts-3.2.0/contracts/access/AccessControl.sol
LLM: Floating Pragma and Outdated Compiler Version
GT: ['SWC-103-Floating Pragma', 'SWC-102-Outdated Compiler Version']


--- Rodando para idx 203 ---
File: DAppSCAN-main/DAppSCAN-source/contracts/openzeppelin-EtherCamp_Hacker_Gold/virtual-accelerator-2529ffe5efd5294b44f1bc89dc9a4721a7b16409/HackerG

In [15]:
comparison_results = []

for bug_pred, gt_cat in tqdm(zip(error_cases['bug_type'], gt_categories), total=len(gt_categories)):
    prompt = prompts.create_zeroshot_error_comparison_prompt(bug_pred, gt_cat)
    response = call_openai(MODEL_NAME, prompt)
    try:
        result = json.loads(response)
        same_error = int(result.get("same_error", False))
    except Exception as e:
        print("Erro ao interpretar resposta da LLM:", e)
        same_error = 0 

    comparison_results.append({'actual_has_error': 1, 'predicted_has_error': same_error})

comparison_df = pd.DataFrame(comparison_results)

precision, recall, f1, conf_matrix = evaluate_experiment(comparison_df)
print(f"Precision: {precision:.2f}")
print(f"Recall: {recall:.2f}")
print(f"F1-Score: {f1:.2f}")
print(f"Confusion Matrix:\n{conf_matrix}")

  0%|          | 0/198 [00:00<?, ?it/s]

100%|██████████| 198/198 [03:25<00:00,  1.04s/it]

Precision: 100.00
Recall: 20.71
F1-Score: 34.31
Confusion Matrix:
[[  0   0]
 [157  41]]
